In [1]:
# ================================================
# Deping 프로젝트 — KOBIS API로 한국 관객수 추가
# 파일명: 02_add_kobis_audience.ipynb
# ================================================

import requests
import json
from pathlib import Path
from tqdm import tqdm
import time
from dotenv import load_dotenv
import os

# 1. .env 로드
load_dotenv()
KOBIS_API_KEY = os.getenv("KOBIS_API_KEY")

if not KOBIS_API_KEY:
    raise ValueError("❌ .env에 KOBIS_API_KEY가 없습니다! 키를 발급받아 추가해주세요.")

print("✅ KOBIS API Key 로드 완료")

# 2. 저장 폴더 생성
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# 3. 기존 JSON 파일 읽기
input_file = PROCESSED_DIR / "tmdb_movies_kr_with_credits_keywords.json"

if not input_file.exists():
    raise FileNotFoundError(f"❌ {input_file} 파일이 없습니다. 01_tmdb_collect.ipynb를 먼저 실행해주세요.")

with open(input_file, "r", encoding="utf-8") as f:
    movies = json.load(f)

print(f"✅ {len(movies)}개 영화 데이터를 불러왔습니다.")

# 4. KOBIS API로 관객수 가져오는 함수
def get_kobis_audience(title_ko: str, release_date: str = None):
    """KOBIS에서 영화 검색 → 누적 관객수(audiAcc) 반환"""
    # Step 1: 영화 검색
    url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieList.json"
    params = {
        "key": KOBIS_API_KEY,
        "movieNm": title_ko,
        "itemPerPage": 10
    }
    
    try:
        resp = requests.get(url, params=params, timeout=10)
        if resp.status_code != 200:
            return None
        
        data = resp.json()
        movie_list = data.get("movieListResult", {}).get("movieList", [])
        
        if not movie_list:
            return None
        
        # Step 2: 개봉일이 가장 비슷한 영화 선택 (정확도 ↑)
        best_match = None
        best_score = -1
        
        for movie in movie_list:
            prdt_year = movie.get("prdtYear", "")
            if release_date and release_date.startswith(prdt_year):
                score = 100
            else:
                score = 50
            if score > best_score:
                best_score = score
                best_match = movie
        
        if not best_match:
            return None
        
        movie_cd = best_match["movieCd"]
        
        # Step 3: 영화 상세정보에서 누적 관객수 가져오기
        detail_url = "http://www.kobis.or.kr/kobisopenapi/webservice/rest/movie/searchMovieInfo.json"
        detail_params = {"key": KOBIS_API_KEY, "movieCd": movie_cd}
        
        detail_resp = requests.get(detail_url, params=detail_params, timeout=10)
        if detail_resp.status_code == 200:
            detail = detail_resp.json()
            audi_acc = detail.get("movieInfoResult", {}).get("movieInfo", {}).get("audiAcc")
            return int(audi_acc) if audi_acc and audi_acc.isdigit() else None
        return None
        
    except Exception as e:
        print(f"⚠️ API 오류 ({title_ko}): {e}")
        return None

# 5. 관객수 일괄 추가
print(f"\n🚀 {len(movies)}개 영화에 대해 KOBIS 관객수 보강 시작...")

for movie in tqdm(movies, desc="KOBIS 관객수 추가"):
    if "audience_kr" not in movie:  # 이미 추가된 영화는 스킵
        audience = get_kobis_audience(
            title_ko=movie.get("title_ko"),
            release_date=movie.get("release_date")
        )
        movie["audience_kr"] = audience
        time.sleep(0.3)  # Rate limit 안전 (하루 3,000회 제한)

# 6. 결과 저장
output_file = PROCESSED_DIR / "tmdb_movies_kr_with_audience.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(movies, f, ensure_ascii=False, indent=2)

print("\n🎉 완료!")
print(f"   • 관객수가 추가된 영화 수: {sum(1 for m in movies if m.get('audience_kr') is not None)}개")
print(f"   • 저장 파일: {output_file.resolve()}")

✅ KOBIS API Key 로드 완료
✅ 240개 영화 데이터를 불러왔습니다.

🚀 240개 영화에 대해 KOBIS 관객수 보강 시작...


KOBIS 관객수 추가: 100%|██████████| 240/240 [04:23<00:00,  1.10s/it]


🎉 완료!
   • 관객수가 추가된 영화 수: 0개
   • 저장 파일: D:\deping\notebooks\data\processed\tmdb_movies_kr_with_audience.json
